# Introduction to Lakehouse runtime Hive catalog

This notebook showcases the Lakehouse runtime Hive catalog with a minimum viable sample.

## 1. Variable and configurations

Configure environment variables. Provide your project ID and a [region](https://cloud.google.com/bigquery/docs/locations#regions) to store your resources, such as `us-central1`.

In [ ]:
PROJECT_ID_LIST=!gcloud config list --format "value(core.project)" 2>/dev/null
PROJECT_ID=PROJECT_ID_LIST[0]
PROJECT_NBR= ! gcloud projects describe $PROJECT_ID | grep projectNumber | cut -d':' -f2 | xargs
PROJECT_NBR=PROJECT_NBR[0]
LOCATION = "us-central1"
STAGE_BUCKET_NAME = f"froyo-staging-{PROJECT_NBR}"
LAKEHOUSE_BUCKET_NAME = f"froyo-lakehouse-hive-{PROJECT_NBR}"
HIVE_CATALOG_NAME="froyo_hive_catalog"
APP_NAME="froyo_app"

## 2. Create a warehouse bucket

In [ ]:
! gsutil mb -p {PROJECT_ID} -l {LOCATION} gs://{LAKEHOUSE_BUCKET_NAME}

## 3. Create the Hive catalog in Lakehouse runtime catalog service

In [ ]:
!gcloud alpha biglake hive catalogs create {HIVE_CATALOG_NAME} --location-uri=gs://{LAKEHOUSE_BUCKET_NAME} --primary-location={LOCATION} --description="froyo hive catalog" --project={PROJECT_ID}

## 4. Create a Spark session with the Hive catalog configuration

In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session


import os
os.environ['DATAPROC_SPARK_CONNECT_DEFAULT_DATASOURCE'] = ""

session = Session()
session.runtime_config.properties = {
 "spark.hive.metastore.blms.project.id": PROJECT_ID,
 "spark.hive.metastore.blms.catalog.default": HIVE_CATALOG_NAME,
 "spark.hive.metastore.warehouse.dir": LAKEHOUSE_BUCKET_NAME,
 "spark.hive.metastore.client.factory.class": "com.google.cloud.bigquery.metastore.client.BigLakeMetastoreClientFactory",
 "spark.sql.catalogImplementation": "hive"
}


spark = DataprocSparkSession.builder.dataprocSessionConfig(session).getOrCreate()
print("Spark session created successfully")

## 5. Create an database

In [ ]:
spark.sql("SHOW databases;").show(truncate=False)

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS froyo_db;")
spark.sql(f"USE froyo_db;")

In [ ]:
spark.sql("SHOW databases;").show(truncate=False)

In [ ]:
spark.sql("SHOW TABLES IN froyo_db").show(truncate=False)

## 6. Read parquet & write to lakehouse bronze layer with table registration in Hive Catalog


In [ ]:
# Load parquet data from GCS staging bucket and persist to bronze (raw) layer with data in full fidelity
customer_stage_df = spark.read.format("parquet").option("inferschema",True).load(f"gs://{STAGE_BUCKET_NAME}/froyo-data/customers")
customer_stage_df.show(2, truncate=False)

In [ ]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "134217728") #128 MB

# Write to bronze layer / raw layer
# Coalescing as there are v v small files
customer_stage_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .partitionBy("region_id") \
    .option("path", f"gs://{LAKEHOUSE_BUCKET_NAME}/froyo-raw/bronze/customer_master") \
    .saveAsTable("froyo_db.b_customer_master")


In [ ]:
# Run some quick stats
spark.sql("select distinct status, count(*) customers from froyo_db.b_customer_master group by status").show(truncate=False)
spark.sql("select count(*) customers from froyo_db.b_customer_master").show(truncate=False)
spark.sql("select count(distinct *) distinct_customers from froyo_db.b_customer_master").show(truncate=False)
spark.sql("select * from froyo_db.b_customer_master limit 2").show(truncate=False)
spark.sql("select distinct region_id, count(*) customers from froyo_db.b_customer_master group by region_id").show(truncate=False)


## 7. Review details of database objects

In [ ]:
spark.sql("SHOW CATALOGS").show(truncate=False)

In [ ]:
spark.sql("SHOW DATABASES").show(truncate=False)

In [ ]:
spark.sql("DESCRIBE DATABASE froyo_db").show(truncate=False)

In [ ]:
spark.sql("SHOW TABLES IN froyo_db").show(truncate=False)

In [ ]:
spark.sql("DESCRIBE FORMATTED froyo_db.b_customer_master").show(40,truncate=False)

## 8. Review the hive warehouse layout in Google Cloud Storage

In [ ]:
import pandas as pd
from google.cloud import storage

# Initialize a GCS client
storage_client = storage.Client(project=PROJECT_ID)

# Get the bucket
bucket = storage_client.get_bucket(LAKEHOUSE_BUCKET_NAME)

# List all blobs in the bucket
blobs = bucket.list_blobs()

# Create a list to store blob information
blob_data = []
for blob in blobs:
    blob_data.append({
        'Name': blob.name,
        'Size (bytes)': blob.size,
        'Content Type': blob.content_type,
        'Creation Time': blob.time_created,
        'Updated Time': blob.updated
    })

# Create a Pandas DataFrame from the blob data
df_gcs_objects = pd.DataFrame(blob_data)

# Display the DataFrame
print(df_gcs_objects.to_markdown(index=False))

## 9. Query the table from BigQuery

You have to use the notation `PROJECT_ID.CATALOG_NAME.DATABASE_NAME.TABLE_NAME`

In [ ]:
%%bigquery --project {PROJECT_ID} --location {LOCATION}

--SELECT * FROM `lakehouse-solutions-build.spark_catalog.froyo_db.b_customer_master` LIMIT 3

SELECT * FROM `lakehouse-solutions-build.froyo_hive_catalog.froyo_db.b_customer_master` LIMIT 3

# This concludes the tutorial. Proceed back to the lab manual.